In [1]:
import os
import json
import copy
import shutil
import itertools
from collections import defaultdict

import numpy as np
from sklearn.model_selection import KFold, GroupKFold

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import (
    DatasetCatalog,
    MetadataCatalog,
    build_detection_test_loader,
)
from detectron2.data.datasets import register_coco_instances


In [2]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
COCO_JSON_PATH = "model2.5_data/train.json"  
IMAGE_ROOT = "/home/mellanie/Desktop/Annotated_dataset/"
CATEGORY_MAP = {"root_hair": 1}
THING_CLASSES = ["root_hair"]

K = 4                    # number of folds
SEARCH_MAX_ITER = 2000   # short runs for search; retrain full-length on winner
SEED = 42

SPLIT_DIR = "kfold_splits"
WORK_DIR = "kfold_runs"

# First pass
# Hyperparameter grid
# HYPERPARAM_GRID = {
#     "SOLVER.BASE_LR": [0.00025, 0.0005],
#     "MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE": [128, 256],
#     "SOLVER.IMS_PER_BATCH": [2],
# }

# Seccond pass (August 10th)
HYPERPARAM_GRID = {
    "SOLVER.BASE_LR": [0.001],
    "MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE": [512],
    "SOLVER.IMS_PER_BATCH": [2],
}


In [3]:

# ---------------------------------------------------------------------------
# Step 1: build K image-grouped folds from one COCO JSON
# ---------------------------------------------------------------------------
def make_kfold_splits(coco_json_path, k=K, seed=SEED, group_key=None):
    """
    Split a COCO JSON into k train/val folds, grouped by image so all
    annotations for one image stay together.

    group_key: optional function(image_dict) -> group id, e.g. to keep
    all images from the same ecotype/plate together in one fold
    (use GroupKFold). If None, plain KFold on image ids is used.
    """
    with open(coco_json_path) as f:
        coco = json.load(f)

    images = coco["images"]
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    image_ids = np.array([img["id"] for img in images])

    if group_key is not None:
        groups = np.array([group_key(img) for img in images])
        splitter = GroupKFold(n_splits=k)
        split_iter = splitter.split(image_ids, groups=groups)
    else:
        splitter = KFold(n_splits=k, shuffle=True, random_state=seed)
        split_iter = splitter.split(image_ids)

    os.makedirs(SPLIT_DIR, exist_ok=True)
    fold_paths = []

    for fold_idx, (train_idx, val_idx) in enumerate(split_iter):
        train_ids = set(image_ids[train_idx].tolist())
        val_ids = set(image_ids[val_idx].tolist())

        for name, id_set in [("train", train_ids), ("val", val_ids)]:
            sub_images = [img for img in images if img["id"] in id_set]
            sub_anns = [a for a in coco["annotations"] if a["image_id"] in id_set]
            sub = {
                "images": sub_images,
                "annotations": sub_anns,
                "categories": coco["categories"],
            }
            out_path = os.path.join(SPLIT_DIR, f"fold{fold_idx}_{name}.json")
            with open(out_path, "w") as f:
                json.dump(sub, f)

        fold_paths.append(
            (
                os.path.join(SPLIT_DIR, f"fold{fold_idx}_train.json"),
                os.path.join(SPLIT_DIR, f"fold{fold_idx}_val.json"),
            )
        )

    return fold_paths


In [4]:

# ---------------------------------------------------------------------------
# Step 2: register / cleanly deregister datasets per fold
# ---------------------------------------------------------------------------
def register_fold(fold_idx, train_json, val_json):
    train_name = f"fold{fold_idx}_train"
    val_name = f"fold{fold_idx}_val"

    for name, json_path in [(train_name, train_json), (val_name, val_json)]:
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)
            MetadataCatalog.remove(name)
        register_coco_instances(name, {}, json_path, IMAGE_ROOT)
        MetadataCatalog.get(name).set(thing_classes=THING_CLASSES)

    return train_name, val_name


def deregister_fold(train_name, val_name):
    for name in [train_name, val_name]:
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)
            MetadataCatalog.remove(name)



In [5]:

# ---------------------------------------------------------------------------
# Step 3: build cfg for a given hyperparameter combo + fold
# ---------------------------------------------------------------------------
class MyTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(
            dataset_name=dataset_name,
            output_dir=output_folder,
            distributed=False,
            use_fast_impl=True,
            tasks=["bbox", "segm"],
        )


def build_cfg(train_name, val_name, hyperparams, output_dir):
    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml")
    )
    cfg.DATASETS.TRAIN = (train_name,)
    cfg.DATASETS.TEST = (val_name,)
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
        "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"
    )
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.MASK_ON = True
    cfg.MODEL.DEVICE = "cuda"
    cfg.SOLVER.MAX_ITER = SEARCH_MAX_ITER
    cfg.SOLVER.CHECKPOINT_PERIOD = SEARCH_MAX_ITER  # only save final checkpoint
    cfg.OUTPUT_DIR = output_dir

    # apply the swept hyperparameters
    for dotted_key, value in hyperparams.items():
        node = cfg
        *path, leaf = dotted_key.split(".")
        for p in path:
            node = getattr(node, p)
        setattr(node, leaf, value)

    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg


In [6]:
# ---------------------------------------------------------------------------
# Step 4: train + evaluate one fold for one hyperparameter combo
# ---------------------------------------------------------------------------
def train_and_eval_fold(fold_idx, train_json, val_json, hyperparams, combo_idx):
    train_name, val_name = register_fold(fold_idx, train_json, val_json)
    run_dir = os.path.join(WORK_DIR, f"combo{combo_idx}_fold{fold_idx}")

    cfg = build_cfg(train_name, val_name, hyperparams, run_dir)

    trainer = MyTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

    evaluator = MyTrainer.build_evaluator(cfg, val_name)
    val_loader = build_detection_test_loader(cfg, val_name)
    results = inference_on_dataset(trainer.model, val_loader, evaluator)

    deregister_fold(train_name, val_name)

    # keep only the metric(s) you care about; segm AP is the natural pick
    # for an instance segmentation task
    segm_ap = results.get("segm", {}).get("AP", float("nan"))
    return segm_ap


In [7]:

# ---------------------------------------------------------------------------
# Step 5: full grid x fold loop
# ---------------------------------------------------------------------------
def run_search():
    fold_paths = make_kfold_splits(COCO_JSON_PATH, k=K)

    keys = list(HYPERPARAM_GRID.keys())
    value_lists = list(HYPERPARAM_GRID.values())
    combos = list(itertools.product(*value_lists))

    all_results = []  # list of dicts: {combo, fold_scores, mean, std}

    for combo_idx, combo_values in enumerate(combos):
        hyperparams = dict(zip(keys, combo_values))
        fold_scores = []

        for fold_idx, (train_json, val_json) in enumerate(fold_paths):
            ap = train_and_eval_fold(fold_idx, train_json, val_json, hyperparams, combo_idx)
            fold_scores.append(ap)
            print(f"[combo {combo_idx} | fold {fold_idx}] {hyperparams} -> segm AP = {ap:.3f}")

        mean_ap = float(np.nanmean(fold_scores))
        std_ap = float(np.nanstd(fold_scores))
        all_results.append(
            {
                "hyperparams": hyperparams,
                "fold_scores": fold_scores,
                "mean_ap": mean_ap,
                "std_ap": std_ap,
            }
        )
        print(f"== combo {combo_idx} summary: mean AP = {mean_ap:.3f} +- {std_ap:.3f} ==\n")

    all_results.sort(key=lambda r: r["mean_ap"], reverse=True)

    print("\nRanked results:")
    for r in all_results:
        print(f"{r['mean_ap']:.3f} +- {r['std_ap']:.3f}  {r['hyperparams']}")

    with open("kfold_search_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    return all_results

In [8]:
run_search()

[08/11 10:42:23 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (4,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.mask_head.predictor.weight' to the model due to incompatible shapes: (80, 256, 1, 1) in the checkpoint but (1, 256, 1, 1) in

[08/11 10:42:24 d2.engine.train_loop]: Starting training from iteration 0


/home/mellanie/.local/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[08/11 10:42:32 d2.utils.events]:  eta: 0:08:45  iter: 19  total_loss: 4.799  loss_cls: 0.6363  loss_box_reg: 0.3106  loss_mask: 0.6903  loss_rpn_cls: 2.587  loss_rpn_loc: 0.5882    time: 0.2733  last_time: 0.3251  data_time: 0.0269  last_data_time: 0.0057   lr: 1.9981e-05  max_mem: 3289M
[08/11 10:42:38 d2.utils.events]:  eta: 0:08:25  iter: 39  total_loss: 2.143  loss_cls: 0.5263  loss_box_reg: 0.3151  loss_mask: 0.6841  loss_rpn_cls: 0.126  loss_rpn_loc: 0.4779    time: 0.2652  last_time: 0.2676  data_time: 0.0071  last_data_time: 0.0089   lr: 3.9961e-05  max_mem: 3289M
[08/11 10:42:43 d2.utils.events]:  eta: 0:08:16  iter: 59  total_loss: 1.996  loss_cls: 0.4272  loss_box_reg: 0.3956  loss_mask: 0.6601  loss_rpn_cls: 0.09169  loss_rpn_loc: 0.43    time: 0.2616  last_time: 0.2547  data_time: 0.0063  last_data_time: 0.0075   lr: 5.9941e-05  max_mem: 3345M
[08/11 10:42:48 d2.utils.events]:  eta: 0:08:08  iter: 79  total_loss: 2.047  loss_cls: 0.4148  loss_box_reg: 0.5007  loss_mask: 0

...

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (4,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.mask_head.predictor.weight' to the model due to incompatible shapes: (80, 256, 1, 1) in the checkpoint but (1, 256, 1, 1) in

[08/11 11:06:10 d2.engine.train_loop]: Starting training from iteration 0
[08/11 11:06:15 d2.utils.events]:  eta: 0:07:17  iter: 19  total_loss: 5.108  loss_cls: 0.7776  loss_box_reg: 0.3472  loss_mask: 0.6937  loss_rpn_cls: 2.679  loss_rpn_loc: 0.6054    time: 0.2209  last_time: 0.2159  data_time: 0.0171  last_data_time: 0.0037   lr: 1.9981e-05  max_mem: 4333M
[08/11 11:06:19 d2.utils.events]:  eta: 0:07:10  iter: 39  total_loss: 2.24  loss_cls: 0.489  loss_box_reg: 0.3319  loss_mask: 0.6829  loss_rpn_cls: 0.1892  loss_rpn_loc: 0.4811    time: 0.2176  last_time: 0.2299  data_time: 0.0042  last_data_time: 0.0042   lr: 3.9961e-05  max_mem: 4333M
[08/11 11:06:24 d2.utils.events]:  eta: 0:07:03  iter: 59  total_loss: 1.89  loss_cls: 0.3687  loss_box_reg: 0.3196  loss_mask: 0.6609  loss_rpn_cls: 0.1009  loss_rpn_loc: 0.411    time: 0.2166  last_time: 0.2575  data_time: 0.0045  last_data_time: 0.0049   lr: 5.9941e-05  max_mem: 4333M
[08/11 11:06:28 d2.utils.events]:  eta: 0:07:01  iter: 79 

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (4,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.mask_head.predictor.weight' to the model due to incompatible shapes: (80, 256, 1, 1) in the checkpoint but (1, 256, 1, 1) in

[08/11 11:17:02 d2.engine.train_loop]: Starting training from iteration 0
[08/11 11:17:07 d2.utils.events]:  eta: 0:07:05  iter: 19  total_loss: 5.609  loss_cls: 0.6859  loss_box_reg: 0.3125  loss_mask: 0.6897  loss_rpn_cls: 3.19  loss_rpn_loc: 0.617    time: 0.2099  last_time: 0.1648  data_time: 0.0195  last_data_time: 0.0038   lr: 1.9981e-05  max_mem: 4442M
[08/11 11:17:11 d2.utils.events]:  eta: 0:06:53  iter: 39  total_loss: 2.133  loss_cls: 0.4835  loss_box_reg: 0.2607  loss_mask: 0.6814  loss_rpn_cls: 0.161  loss_rpn_loc: 0.4866    time: 0.2069  last_time: 0.2142  data_time: 0.0039  last_data_time: 0.0037   lr: 3.9961e-05  max_mem: 4442M
[08/11 11:17:15 d2.utils.events]:  eta: 0:06:49  iter: 59  total_loss: 1.907  loss_cls: 0.3775  loss_box_reg: 0.3818  loss_mask: 0.66  loss_rpn_cls: 0.07919  loss_rpn_loc: 0.4187    time: 0.2072  last_time: 0.2066  data_time: 0.0041  last_data_time: 0.0041   lr: 5.9941e-05  max_mem: 4442M
[08/11 11:17:20 d2.utils.events]:  eta: 0:06:50  iter: 79 

[{'hyperparams': {'SOLVER.BASE_LR': 0.001,
   'MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE': 512,
   'SOLVER.IMS_PER_BATCH': 2},
  'fold_scores': [19.550341754772273,
   23.79608616205143,
   22.039820791746454,
   23.00937629903594],
  'mean_ap': 22.09890625190152,
  'std_ap': 1.5975017560064528}]